## 한국문화 멀티모달 질의응답 데이터: 전처리 및 EDA

국립국어원 인공지능(AI)말평의 **한국문화 멀티모달 질의응답** 데이터 2,000문항을 점검한다.

목표:

1. JSON 스키마와 이미지 연결 무결성 검증
2. split·문항 유형·텍스트 길이·답안 형식 분석
3. 이미지 크기·해상도·파일 형식 분석
4. 반복 질문 템플릿과 교차 split 동일 이미지 점검
5. 원본을 수정하지 않는 정규화 파생 데이터 생성

> 주의: 질문 키워드 기반 OCR/부정형/복수정답 표시는 휴리스틱이며 정답 라벨이 아니다. 자동 탐지 결과는 삭제 기준이 아니라 수동 검토 큐로 사용한다.

In [1]:
import os
import sys
import json
import re
import hashlib
import unicodedata
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".cache" / "matplotlib"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from PIL import Image
from IPython.display import Markdown, display

DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
PROCESSED_ROOT = OUTPUT_ROOT / "processed"
EDA_ROOT = OUTPUT_ROOT / "eda"
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
EDA_ROOT.mkdir(parents=True, exist_ok=True)

SPLITS = ["train", "validation", "test"]
FORMS = ["MC", "SA", "LA"]
EXPECTED_COUNTS = {"train": 1000, "validation": 200, "test": 800}
COMPUTE_SHA256 = True  # 8.4 GiB 전체를 읽는다. 빠른 재실행 시 False로 변경 가능

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid", context="notebook")
print(f"Python: {sys.version.split()[0]}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Data root exists: {DATA_ROOT.exists()}")

Python: 3.12.5
Project root: /Users/hyejin/AI_CV
Data root exists: True


### 1. 원본 JSON 탐색과 기본 구조 확인

폴더명이 macOS에서 NFD 유니코드로 저장되어 있을 수 있으므로 한글 폴더명을 직접 하드코딩하지 않고 split 접미사로 JSON을 탐색한다.

In [2]:
def find_split_json(split: str) -> Path:
    candidates = sorted(DATA_ROOT.rglob(f"*_{split}.json"))
    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Expected one JSON for {split}, found {len(candidates)}: {candidates}"
        )
    return candidates[0]


json_paths = {split: find_split_json(split) for split in SPLITS}
raw_by_split = {
    split: json.loads(path.read_text(encoding="utf-8"))
    for split, path in json_paths.items()
}

overview = pd.DataFrame(
    [
        {
            "split": split,
            "json_path": str(json_paths[split].relative_to(PROJECT_ROOT)),
            "records": len(raw_by_split[split]),
            "expected": EXPECTED_COUNTS[split],
            "count_match": len(raw_by_split[split]) == EXPECTED_COUNTS[split],
        }
        for split in SPLITS
    ]
)
display(overview)
display(raw_by_split["train"][0])

,split,json_path,records,expected,count_match
0,train,data/한국문화 멀티모달 질의응답/한국문화 멀티모달 질의응답_train.json,1000,1000,True
1,validation,data/한국문화 멀티모달 질의응답/한국문화 멀티모달 질의응답_validation.json,200,200,True
2,test,data/한국문화 멀티모달 질의응답/한국문화 멀티모달 질의응답_test.json,800,800,True


{'metadata': {'question_id': '0083',
  'task_type': '한국문화 멀티모달 질의응답',
  'corpus_name': '한국문화 멀티모달 질의응답',
  'split': 'train',
  'question_form': 'MC'},
 'model_input': {'image_name': '0083.jpg',
  'question': '그림 속 여성이 입고 있는 옷에 대한 설명이 틀린 것은 어느 것인지 고르시오.',
  'options': ['1) 소매에 흰색 부분이 달려 있다.',
   '2) 주로 붉은색 비단으로 만들어졌다.',
   '3) 치마 아랫단에는 장수와 복을 상징하는 글자가 금박으로 새겨져 있다.',
   '4) 봉황과 모란이 수놓아져 있다.',
   '5) 소맷단에 있는 무늬들은 모두 천으로 만들어 붙인 것이다.']},
 'model_output': {'answer': '5'}}

### 2. 손실 없는 텍스트 정규화와 평탄화

- 모든 문자열을 Unicode NFC로 통일한다.
- 앞뒤 공백과 연속 공백을 정리하되 의미가 있는 문장부호는 유지한다.
- 원문 답안은 유지하고 MC 선택지 번호 집합을 별도 파생 변수로 만든다.
- test에는 공개 정답이 없으므로 `answer=None`으로 둔다.

In [3]:
MC_CHOICE_RE = re.compile(r"(?<!\d)([1-5])(?!\d)")
LENGTH_RE = re.compile(r"(?P<length>\d+)\s*(?P<unit>음절|글자|어절|자)(?:로|으로)?")
OCR_RE = re.compile(r"적혀|쓰여|써 있|문구|글자|안내문|안내판|표지판|간판|현수막|포스터|메뉴판|명판|읽(?:고|어|으)")
NEGATION_RE = re.compile(r"옳지 않|틀린|아닌|않은|제외|수 없는|해당하지 않")
MULTI_RE = re.compile(r"모두\s*(?:고르|답|찾)|전부\s*(?:고르|답|찾)|각각|복수")


def normalize_text(value) -> str:
    text = unicodedata.normalize("NFC", "" if value is None else str(value))
    return re.sub(r"\s+", " ", text).strip()


def visible_char_count(text: str) -> int:
    return sum(char.isalnum() for char in text)


def extract_mc_choices(answer: str) -> list[int]:
    return sorted({int(choice) for choice in MC_CHOICE_RE.findall(answer or "")})


def parse_length_constraint(question: str):
    match = LENGTH_RE.search(question)
    if not match:
        return None, None
    return int(match.group("length")), match.group("unit")


rows = []
schema_issues = []
for split in SPLITS:
    for position, raw in enumerate(raw_by_split[split]):
        metadata = raw.get("metadata", {})
        model_input = raw.get("model_input", {})
        model_output = raw.get("model_output")
        question_id = normalize_text(metadata.get("question_id"))
        question = normalize_text(model_input.get("question"))
        image_name = normalize_text(model_input.get("image_name"))
        question_form = normalize_text(metadata.get("question_form")).upper()
        options = [normalize_text(option) for option in (model_input.get("options") or [])]
        answer = None if model_output is None else normalize_text(model_output.get("answer"))
        requested_length, requested_unit = parse_length_constraint(question)

        row = {
            "global_id": f"{split}:{question_id}",
            "question_id": question_id,
            "split": split,
            "question_form": question_form,
            "image_name": image_name,
            "image_path": DATA_ROOT / split / image_name,
            "question": question,
            "options": options,
            "option_count": len(options),
            "answer": answer,
            "mc_choices": extract_mc_choices(answer) if question_form == "MC" else [],
            "question_chars": len(question),
            "question_eojeol": len(question.split()),
            "answer_chars": None if answer is None else len(answer),
            "answer_visible_chars": None if answer is None else visible_char_count(answer),
            "requested_length": requested_length,
            "requested_unit": requested_unit,
            "ocr_cue": bool(OCR_RE.search(question)),
            "negation_cue": bool(NEGATION_RE.search(question)),
            "multi_answer_cue": bool(MULTI_RE.search(question)),
        }
        rows.append(row)

        if metadata.get("split") != split:
            schema_issues.append((row["global_id"], "split_mismatch"))
        if question_form not in FORMS:
            schema_issues.append((row["global_id"], "unknown_question_form"))
        if question_form == "MC" and len(options) != 5:
            schema_issues.append((row["global_id"], "mc_option_count_not_5"))
        if question_form != "MC" and options:
            schema_issues.append((row["global_id"], "non_mc_has_options"))
        if split != "test" and answer is None:
            schema_issues.append((row["global_id"], "missing_answer"))
        if split == "test" and answer is not None:
            schema_issues.append((row["global_id"], "test_answer_present"))

df = pd.DataFrame(rows)
print(f"Records: {len(df):,}")
print(f"Schema issues: {len(schema_issues):,}")
display(df.head(3))

Records: 2,000
Schema issues: 0


,global_id,question_id,split,question_form,image_name,image_path,question,options,option_count,answer,mc_choices,question_chars,question_eojeol,answer_chars,answer_visible_chars,requested_length,requested_unit,ocr_cue,negation_cue,multi_answer_cue
0,train:0083,0083,train,MC,0083.jpg,/Users/hyejin/AI_CV/data/train/0083.jpg,그림 속 여성이 입고 있는 옷에 대한 설명이 틀린 것은 어느 것인지 고르시오.,"[1) 소매에 흰색 부분이 달려 있다., 2) 주로 붉은색 비단으로 만들어졌다., 3) 치마 아랫단에는 장수와 복을 상징하는 글자가 금박으로 새겨져 있다., 4) 봉황과 모란이 수놓아져 있다., 5) 소맷단에...",5,5,[5],43,13,1.0,1.0,NaN,None,False,True,False
1,train:0771,0771,train,MC,P30909.jpg,/Users/hyejin/AI_CV/data/train/P30909.jpg,사진 속 장소와 그와 관련된 내용에 관한 설명으로 옳지 않은 것을 고르시오.,"[1) 미국은 6) 25 전쟁에 참전했다., 2) 이미지 속 물체는 미국 참전을 기념하는 기념비이다., 3) 기념비에는 미국 참전에 관한 내용이 영어로 적혀 있다., 4) 해당 전쟁은 1952년도에 발발한 전...",5,4,[4],42,12,1.0,1.0,NaN,None,False,True,False
2,train:1798,1798,train,MC,P11364.jpg,/Users/hyejin/AI_CV/data/train/P11364.jpg,사진 속 과일 중 제사상에 올릴 수 없는 과일을 고르시오.,"[1) 전부 올릴 수 있음, 2) 사과, 3) 자두, 4) 샤인머스켓, 5) 거봉]",5,1,[1],32,10,1.0,1.0,NaN,None,False,True,False


### 3. split 및 문항 유형 분포

In [4]:
split_form = (
    df.groupby(["split", "question_form"])
      .size()
      .unstack(fill_value=0)
      .reindex(index=SPLITS, columns=FORMS, fill_value=0)
)
display(split_form)

plot_df = split_form.reset_index().melt(
    id_vars="split", var_name="question_form", value_name="count"
)
fig, ax = plt.subplots(figsize=(9, 4.8))
sns.barplot(
    data=plot_df, x="split", y="count", hue="question_form",
    order=SPLITS, hue_order=FORMS, palette="colorblind", ax=ax
)
for container in ax.containers:
    ax.bar_label(container, padding=2, fontsize=9)
ax.set(title="Question mix by split", xlabel="Split", ylabel="Questions")
ax.legend(title="Form")
plt.tight_layout()
fig.savefig(EDA_ROOT / "question_mix.png", dpi=180, bbox_inches="tight")
plt.show()

question_form,MC,SA,LA
split,,,
train,518,254,228
validation,103,52,45
test,415,206,179


/var/folders/xg/zxhx43fx7178bbhgcx6ksw0c0000gn/T/ipykernel_27325/21038187.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4. 질문 및 정답 길이

In [5]:
labeled = df[df["answer"].notna()].copy()

text_summary = pd.DataFrame(
    {
        "question_chars": df["question_chars"].describe(percentiles=[0.25, 0.5, 0.75, 0.95]),
        "answer_chars_labeled": labeled["answer_chars"].describe(percentiles=[0.25, 0.5, 0.75, 0.95]),
    }
)
display(text_summary)

answer_by_form = (
    labeled.groupby("question_form")["answer_chars"]
    .describe(percentiles=[0.5, 0.95])
    .reindex(FORMS)
)
display(answer_by_form)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
sns.boxplot(
    data=df, x="question_form", y="question_chars", order=FORMS,
    hue="question_form", palette="colorblind", legend=False,
    showfliers=False, ax=axes[0]
)
axes[0].set(title="Question length", xlabel="Form", ylabel="Characters (outliers hidden)")
sns.boxplot(
    data=labeled, x="question_form", y="answer_chars", order=FORMS,
    hue="question_form", palette="colorblind", legend=False,
    showfliers=False, ax=axes[1]
)
axes[1].set(title="Answer length: train + validation", xlabel="Form", ylabel="Characters (outliers hidden)")
plt.tight_layout()
fig.savefig(EDA_ROOT / "text_lengths.png", dpi=180, bbox_inches="tight")
plt.show()

,question_chars,answer_chars_labeled
count,2000.000000,1200.000000
mean,42.437500,37.084167
std,16.719655,72.479064
min,18.000000,1.000000
25%,31.000000,1.000000
50%,37.000000,2.000000
75%,51.000000,7.000000
95%,76.000000,207.050000
max,158.000000,338.000000


,count,mean,std,min,50%,95%,max
question_form,,,,,,,
MC,621.0,1.219002,0.674675,1.0,1.0,3.00,5.0
SA,306.0,4.104575,2.858782,1.0,3.0,8.75,22.0
LA,273.0,155.633700,69.859875,27.0,164.0,277.40,338.0


/var/folders/xg/zxhx43fx7178bbhgcx6ksw0c0000gn/T/ipykernel_27325/3306597689.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5. 이미지 무결성·크기·정확 중복

이미지 헤더에서 크기·형식·모드를 읽고, `COMPUTE_SHA256=True`일 때 전체 파일 바이트의 SHA-256을 계산한다. SHA-256 중복은 완전히 같은 파일만 탐지하며, 크롭·리사이즈·압축률이 다른 근접 중복은 탐지하지 않는다.

In [6]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def profile_image(item):
    global_id, path = item
    result = {
        "global_id": global_id,
        "image_found": path.is_file(),
        "image_bytes": None,
        "image_mb": None,
        "image_width": None,
        "image_height": None,
        "image_megapixels": None,
        "image_aspect_ratio": None,
        "image_format": None,
        "image_mode": None,
        "image_sha256": None,
        "image_error": None,
    }
    if not path.is_file():
        return result
    try:
        size = path.stat().st_size
        with Image.open(path) as image:
            width, height = image.size
            result.update(
                image_bytes=size,
                image_mb=size / (1024 * 1024),
                image_width=width,
                image_height=height,
                image_megapixels=width * height / 1_000_000,
                image_aspect_ratio=width / height if height else None,
                image_format=image.format,
                image_mode=image.mode,
            )
        if COMPUTE_SHA256:
            result["image_sha256"] = sha256_file(path)
    except Exception as exc:
        result["image_error"] = f"{type(exc).__name__}: {exc}"
    return result


image_items = list(df[["global_id", "image_path"]].itertuples(index=False, name=None))
with ThreadPoolExecutor(max_workers=min(8, os.cpu_count() or 4)) as executor:
    image_profiles = list(executor.map(profile_image, image_items))

image_df = pd.DataFrame(image_profiles)
df = df.merge(image_df, on="global_id", how="left", validate="one_to_one")

image_summary = df.groupby("split").agg(
    images=("image_found", "size"),
    missing=("image_found", lambda s: (~s).sum()),
    total_gib=("image_bytes", lambda s: s.sum() / 1024**3),
    median_mb=("image_mb", "median"),
    p95_mb=("image_mb", lambda s: s.quantile(0.95)),
    max_mb=("image_mb", "max"),
    median_mp=("image_megapixels", "median"),
    max_mp=("image_megapixels", "max"),
).round(3)
display(image_summary)
display(df["image_format"].value_counts(dropna=False).rename("count").to_frame())

fig, ax = plt.subplots(figsize=(9, 5))
plot_images = df.dropna(subset=["image_megapixels", "image_mb"])
sns.scatterplot(
    data=plot_images, x="image_megapixels", y="image_mb", hue="split",
    hue_order=SPLITS, palette="colorblind", alpha=0.6, s=30, ax=ax
)
ax.set(
    xlim=(0, plot_images["image_megapixels"].quantile(0.99) * 1.05),
    ylim=(0, plot_images["image_mb"].quantile(0.99) * 1.05),
    title="Image compute profile (axes capped at p99)",
    xlabel="Megapixels", ylabel="File size (MiB)",
)
ax.legend(title="Split")
plt.tight_layout()
fig.savefig(EDA_ROOT / "image_profile.png", dpi=180, bbox_inches="tight")
plt.show()

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


,images,missing,total_gib,median_mb,p95_mb,max_mb,median_mp,max_mp
split,,,,,,,,
test,800,0,3.258,3.221,10.926,62.991,12.193,108.000
train,1000,0,4.345,3.393,11.799,63.208,12.193,64.144
validation,200,0,0.785,3.353,10.493,46.834,12.054,36.152


,count
image_format,
JPEG,1760
MPO,238
TIFF,2


/var/folders/xg/zxhx43fx7178bbhgcx6ksw0c0000gn/T/ipykernel_27325/3484333526.py:83: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 6. 반복 질문 문구와 교차 split 동일 이미지

동일한 질문 문구는 일반적인 MC 템플릿 반복일 수 있으므로 곧바로 데이터 누수로 해석하지 않는다. 반면 동일 이미지 해시가 train/validation/test 사이에 나타나는 경우, 같은 시각 정보에 대한 다른 질문이 존재할 수 있어 평가 해석과 retrieval cache 설계에 영향을 준다.

In [7]:
def cross_split_groups(frame: pd.DataFrame, key: str):
    groups = []
    for value, group in frame.dropna(subset=[key]).groupby(key):
        if group["split"].nunique() > 1:
            groups.append((value, group.copy()))
    return groups


question_text_groups = cross_split_groups(df, "question")
image_name_groups = cross_split_groups(df, "image_name")
image_hash_groups = cross_split_groups(df, "image_sha256") if COMPUTE_SHA256 else []

duplicate_overview = pd.DataFrame(
    [
        {"check": "Repeated exact question text", "cross_split_groups": len(question_text_groups), "affected_rows": sum(len(g) for _, g in question_text_groups)},
        {"check": "Repeated image filename", "cross_split_groups": len(image_name_groups), "affected_rows": sum(len(g) for _, g in image_name_groups)},
        {"check": "Identical image SHA-256", "cross_split_groups": len(image_hash_groups), "affected_rows": sum(len(g) for _, g in image_hash_groups)},
    ]
)
display(duplicate_overview)

if image_hash_groups:
    duplicate_image_rows = pd.concat(
        [group.assign(duplicate_hash=hash_value) for hash_value, group in image_hash_groups],
        ignore_index=True,
    )
    display(
        duplicate_image_rows[
            ["global_id", "split", "image_name", "question_form", "question", "answer", "duplicate_hash"]
        ].sort_values("duplicate_hash")
    )

largest_templates = sorted(
    [
        {
            "question": text,
            "rows": len(group),
            "splits": ", ".join(sorted(group["split"].unique())),
        }
        for text, group in question_text_groups
    ],
    key=lambda item: item["rows"], reverse=True,
)[:10]
display(pd.DataFrame(largest_templates))

,check,cross_split_groups,affected_rows
0,Repeated exact question text,39,302
1,Repeated image filename,4,8
2,Identical image SHA-256,6,12


,global_id,split,image_name,question_form,question,answer,duplicate_hash
0,train:1978,train,PK1025.jpg,MC,사진 속 안내문을 통해 알 수 있는 한국 통신사의 마케팅 방식으로 가장 적절한 것을 고르시오.,1,5f8b84b7eb497933341dff0e01e96be419b4b4b76ef2fb6a089fd539975b644b
1,test:1139,test,PK2003.jpg,SA,사진 속 매장 브랜드와 함께 한국의 주요 이동통신사로 한국통신을 전신으로 하는 회사의 이름을 영문 알파벳으로 답하시오.,None,5f8b84b7eb497933341dff0e01e96be419b4b4b76ef2fb6a089fd539975b644b
2,validation:1812,validation,P20191.jpg,MC,사진 속 장소에 대한 설명으로 옳은 것을 고르시오.,1,68fbf71d320ded81928215d9f88694a9e4837741ee011ace9ddcbdc1b49948da
3,test:1941,test,P20191.jpg,MC,제시된 사진을 보고 옳은 것을 고르시오.,None,68fbf71d320ded81928215d9f88694a9e4837741ee011ace9ddcbdc1b49948da
4,train:1648,train,P30164.jpg,MC,다음 중 이 장소에서 가장 적절한 행동은 무엇인가?,1,8ee02884f0c4ea0787f1389510dfefa2129711be905e459fa568beadbd4d5705
5,test:0745,test,P30164.jpg,MC,사진 속 공간에 대한 설명 중 옳지 않은 것을 모두 고르시오.,None,8ee02884f0c4ea0787f1389510dfefa2129711be905e459fa568beadbd4d5705
6,train:1730,train,PK2148.jpg,SA,박스 안에 들어 있는 위생용품을 동시에 볼 수 있는 장소는 어디인지 3음절로 답하시오.,화장실,d3cbb27a7d5290f6128278f71b4c746b1d3275ef1b6fbddd0066b412e10acf24
7,test:1368,test,PK1077.jpg,LA,"사진에 나타난 나누미 점보롤 화장지 포장의 '장애인 직업재활시설' 표시를 읽고, 한국 장애인 직업재활시설 제도의 운영 방식과 생산 물품 우선 구매 제도를 서술하시오.",None,d3cbb27a7d5290f6128278f71b4c746b1d3275ef1b6fbddd0066b412e10acf24
8,train:1992,train,P30290.jpg,LA,사진 속 체험 자료를 참고하여 이 시기 우리나라 여권의 특징을 서술하시오.,"사진 속 체험은 대한제국 시기의 여권을 재현하는 것이다. 우리나라 여권의 시작은 19세기 말 발급된 집조로 조선 중앙 정부기관에서 개인을 대상으로 발급되었다. 성명, 직위, 목적지, 발급일자 등이 기재되어 있다.",e4c13af6d094736caf2fe1e8253b5865f231db09287f2d10ecdfe418417e0f45
9,validation:0754,validation,P30290.jpg,SA,이미지와 관련된 시기 출국 확인 문서를 일컫던 명칭이 무엇인지 2음절로 답하시오.,집조,e4c13af6d094736caf2fe1e8253b5865f231db09287f2d10ecdfe418417e0f45


,question,rows,splits
0,사진 속 장면에 대한 설명 중 옳은 것을 고르시오.,68,"test, train, validation"
1,사진 속 장면에 대한 설명 중 옳지 않은 것을 고르시오.,57,"test, train, validation"
2,사진 속 장면에 대한 설명으로 옳지 않은 것을 고르시오.,24,"test, train, validation"
3,사진 속 장면에 대한 설명 중 옳은 것을 모두 고르시오.,13,"test, train, validation"
4,"사진 속 장면에 대한 설명 중, 옳지 않은 것을 고르시오.",9,"test, train, validation"
5,사진을 보고 옳은 것을 고르시오.,9,"test, train, validation"
6,사진 속 물체는 무엇인지 고르시오.,7,"test, train, validation"
7,다음 사진 속 장소에 대한 설명으로 옳지 않은 것을 고르시오.,6,"test, train, validation"
8,다음 사진에 대한 설명으로 맞는 것을 고르시오.,6,"test, train"
9,사진 속 장면에 대한 설명으로 옳은 것을 고르시오.,6,"test, train"


### 7. 답안 형식 및 품질 위험

In [8]:
def constraint_matches(row):
    if row["answer"] is None or pd.isna(row["requested_length"]):
        return None
    observed = len(row["answer"].split()) if row["requested_unit"] == "어절" else visible_char_count(row["answer"])
    return observed == int(row["requested_length"])


df["length_constraint_match"] = df.apply(constraint_matches, axis=1)
labeled_constraints = df[df["answer"].notna() & df["requested_length"].notna()].copy()
la_over_250 = df[(df["question_form"] == "LA") & (df["answer_chars"] > 250)].copy()
large_images = df[df["image_mb"] > 15].copy()
high_resolution = df[df["image_megapixels"] > 25].copy()

mc_choice_frequency = Counter(
    choice
    for choices in labeled.loc[labeled["question_form"] == "MC", "mc_choices"]
    for choice in choices
)
mc_summary = pd.DataFrame(
    {"choice": range(1, 6), "frequency": [mc_choice_frequency.get(i, 0) for i in range(1, 6)]}
)
display(mc_summary)

quality_summary = pd.DataFrame(
    [
        {"check": "Missing image", "count": int((~df["image_found"]).sum())},
        {"check": "Image open error", "count": int(df["image_error"].notna().sum())},
        {"check": "Image > 15 MiB", "count": len(large_images)},
        {"check": "Image > 25 megapixels", "count": len(high_resolution)},
        {"check": "LA reference answer > 250 chars", "count": len(la_over_250)},
        {"check": "Answer/requested-length mismatch (heuristic)", "count": int((labeled_constraints["length_constraint_match"] == False).sum())},
        {"check": "Cross-split identical image hash groups", "count": len(image_hash_groups)},
    ]
)
display(quality_summary)

if len(la_over_250):
    display(la_over_250[["global_id", "question", "answer_chars", "answer"]].sort_values("answer_chars", ascending=False).head(10))

if len(labeled_constraints):
    mismatch_view = labeled_constraints[labeled_constraints["length_constraint_match"] == False]
    display(mismatch_view[["global_id", "question", "requested_length", "requested_unit", "answer"]].head(20))

,choice,frequency
0,1,153
1,2,142
2,3,137
3,4,138
4,5,119


,check,count
0,Missing image,0
1,Image open error,0
2,Image > 15 MiB,41
3,Image > 25 megapixels,33
4,LA reference answer > 250 chars,26
5,Answer/requested-length mismatch (heuristic),23
6,Cross-split identical image hash groups,6


,global_id,question,answer_chars,answer
152,train:0949,사진 속 연표의 주제가 되는 음식의 한국 내 변천사를 서술하시오.,338.0,"우리나라에서 커피는 19세기 조선에 들어와 처음에는 선교사와 왕실, 외교관, 지식인 등 일부 계층을 중심으로 소비되었다. 개항 이후에는 호텔과 다과점에서 일반인에게도 판매되기 시작했으며, 일제강점기에는 다방이..."
380,train:0675,"저수지 울타리 뒤편에 세워진 표지판의 문구와 안내 표시를 바탕으로, 이곳에서 특정 행위를 제한하는 이유와 목적을 한국의 안전 관리 상식과 연계하여 추론하여 서술하시오.",337.0,사진 속 저수지 주변에는 안전사고를 방지하기 위해 '수영금지'라는 붉은색 통제 문구와 함께 '깊은 수심 주의'라는 노란색 경고 표지판이 설치되어 있습니다. 표지판 하단에 '경기도 소방재난본부장'과 '과천소방서...
903,train:0527,"현충공원 입구에 설치된 파란색 안내판에 적힌 한글 문구와 시각적 요소를 바탕으로, 관람객이 인지해야 할 공간의 성격과 준수해야 할 제한 사항이 무엇인지 사진 속 사실만을 근거로 서술하시오.",332.0,사진 속 현충공원의 계단 진입로에는 공간의 성격과 이용 수칙을 알리는 파란색 안내판이 세워져 있습니다. 안내판 최상단에는 '이곳은 순국선열과 호국영령을 추모하는 공간입니다'라고 명시되어 있어 해당 장소가 지닌...
921,train:0546,"영화관 층별 안내판의 정보와 한국의 일반적인 상영관 이용 상식을 바탕으로, 현재 8층에 있는 관람객이 영화 관람 및 간식 구매를 위해 취해야 할 행동을 논리적으로 추론하여 서술하시오.",315.0,"안내판에 따르면 현재 위치인 8층에는 13관부터 17관까지의 상영관이 위치해 있으므로, 만약 본인이 예매한 상영관이 이 범위에 해당한다면 해당 층에서 바로 입장을 준비하면 됩니다. 반면 8관~12관을 이용하려..."
897,train:0521,"전광판의 열차 운행 정보와 현재 시간 및 한국의 철도 이용 상식을 바탕으로, 이 대합실에 있는 승객이 전광판을 확인한 후 취해야 할 행동과 이동 방향을 논리적으로 추론하여 서술하시오.",315.0,안내 전광판에 표시된 현재 시간은 11시 57분이며 가장 가까운 출발 열차인 14시 10분 누리로 열차까지는 약 2시간 이상의 충분한 시간적 여유가 있는 상황입니다. 따라서 승객은 급하게 승강장으로 이동하기보...
523,train:0819,사진 내 엑스 배너에 제시된 문서를 발급받는 절차와 준비해야 하는 서류를 서술하시오,299.0,여권을 처음 발급받으려면 신청자가 시·군·구청 등 여권사무 대행기관을 직접 방문해야 한다. 먼저 여권발급신청서를 작성하여 필요한 서류와 함께 접수한 뒤 수수료를 납부한다. 이후 여권이 발급되면 신분증을 지참하...
457,train:0840,"테이블 위에 놓인 상자들의 표면 문구과 포장 형태를 확인하고, 이 상품이 주로 어떤 용도로 소비되는지 서술하시오.",298.0,"대리석 모양 테이블 위에 '복분자'라는 제품명이 크게 적힌 붉은색 상자 4개가 나란히 놓여 있으며, 상단에는 '선물용'이라는 문구가 또렷하게 인쇄되어 있다. 또한 상자 최상단은 별도의 쇼핑백 없이도 손쉽게 들..."
451,train:0762,사진에 나타난 장소의 건축적 특징을 서술하시오.,297.0,더현대 서울은 사람들이 자연을 느끼며 머물고 휴식할 수 있는 공간을 지향한다. 외관에는 한국 전통 건축의 자적색 기둥과 단청을 현대적으로 재해석해 적용했으며 실내에는 유리 천장과 보이드 구조를 활용해 자연광이...
905,train:0529,"제품 포장재에 적힌 문구와 예시 그림을 바탕으로, 이 자석 장난감이 어린이들에게 제공하는 구체적인 놀이 방식과 학습 효과를 한국의 영유아 교육 상식에 맞추어 추론하여 서술하시오.",296.0,"이 제품은 자음 28개와 모음 22개라는 수량의 자석 글자로 구성되어 있어, 영유아가 기초적인 한국어 낱말부터 다양한 어휘까지 자유롭게 조합해 보기에 적합하다. 제품 하단에 제시된 '퍼즐 맞추기', '글씨 만..."
657,train:1179,"사진 속 상단 전광판과 표지판에 적힌 문구들을 확인하고, 터널 진입 전 이러한 시설물들이 설치된 목적과 현대 한국의 운전 문화 상식을 연계하여 추론하여 서술하시오.",291.0,"터널 입구 위에 '터널진입차단시설'이라는 대형 전광판이 설치되어 있고, 그 옆으로 '구간단속 시점' 및 제한 속도 '70'을 알리는 표지판이 보입니다. 현대 한국의 도로교통 상식과 연계해 볼 때, 터널진입차단..."


,global_id,question,requested_length,requested_unit,answer
154,train:0229,작품 속 아이들이 가지고 노는 것의 이름과 개수를 차례대로 각각 2음절로 답하시오.,2.0,음절,팽이/3개
186,train:0436,"사진 속 인물의 계급을 1음절로 답하고, 이러한 그림을 무엇이라고 하는지 2음절로 차례대로 답하시오.",1.0,음절,왕/어진
252,train:0388,사진 속 인물이 쓴 모자의 명칭과 이 모자는 어떤 의례에 쓰는지 차례대로 각각 2음절로 답하시오.,2.0,음절,굴건/상례
275,train:0346,"정면으로 보이는 집 작은 창문 앞쪽에 달려 있는 것의 이름과 개수를 차례대로 답하되, 이름은 3음절로 개수는 2음절로 답하시오.",3.0,음절,소쿠리/2개
634,train:0208,"사진의 오른쪽 하단부에는 실로 어떤 행위를 한 결과물이 놓여 있다. 이를 가리키는 단어를 2음절로 쓰고, 이 단어가 포함된 관용구 두 개를 뜻과 함께 서술하시오.",2.0,음절,사진의 오른쪽 하단부에 놓여 있는 것은 ‘매듭’이다. 매듭과 관련된 것으로는 일을 마무리한다는 의미의 ‘매듭을 짓다’와 문제를 해결한다는 뜻을 가진 ‘매듭을 풀다’라는 관용구가 있다.
676,train:0228,"사진의 오른쪽 하단에 앉은 사람이 던진 것의 명칭을 1음절로 답하고, 그 개수는 2음절로 차례대로 답하시오.",1.0,음절,윷/4개
681,train:0383,"이미지 속 인물이 착용한 의복의 명칭은 3음절로, 착용 주체는 1음절로 차례대로 답하시오.",3.0,음절,구장복/왕
784,train:0219,사진 속 악기의 명칭과 이 악기에 매달려 있는 돌의 모양을 닮은 한글 자음을 차례대로 각각 2음절로 답하시오.,2.0,음절,편경/기역
785,train:0235,사진 속에서 불을 밝혀두기 위한 용도로 만들어진 것의 이름과 재료를 각각 2음절과 1음절로 차례대로 답하시오.,2.0,음절,석등/돌
790,train:0248,사진 속의 책은 우리나라와 중국의 의서를 엮어 광해군 2년(1610)에 완성한 책이다. 이 책의 저자와 저자의 관직을 각각 2음절로 순서대로 답하시오.,2.0,음절,허준/의관


### 8. 질문 문구 기반 난이도 신호

OCR 필요도는 실제 이미지를 보지 않고 질문 문구만으로 추정하므로 하한에 가깝다. 수작업 태깅용 표본을 만들 때 우선순위를 정하는 용도로만 사용한다.

In [9]:
cue_columns = ["ocr_cue", "negation_cue", "multi_answer_cue"]
cue_labels = {
    "ocr_cue": "OCR cue",
    "negation_cue": "Negation cue",
    "multi_answer_cue": "Multi-answer cue",
}

cue_summary = pd.DataFrame(
    [
        {
            "cue": cue_labels[cue],
            "count": int(df[cue].sum()),
            "ratio_pct": round(df[cue].mean() * 100, 1),
        }
        for cue in cue_columns
    ]
)
display(cue_summary)

cue_by_form = pd.DataFrame(
    [
        {
            "question_form": form,
            "cue": cue_labels[cue],
            "ratio_pct": df.loc[df["question_form"] == form, cue].mean() * 100,
        }
        for form in FORMS
        for cue in cue_columns
    ]
)
fig, ax = plt.subplots(figsize=(9, 4.8))
sns.barplot(
    data=cue_by_form, x="ratio_pct", y="cue", hue="question_form",
    hue_order=FORMS, palette="colorblind", ax=ax
)
ax.set(title="Question-language risk cues", xlabel="Share within form (%)", ylabel="Heuristic cue")
ax.legend(title="Form")
plt.tight_layout()
fig.savefig(EDA_ROOT / "question_cues.png", dpi=180, bbox_inches="tight")
plt.show()

,cue,count,ratio_pct
0,OCR cue,331,16.6
1,Negation cue,388,19.4
2,Multi-answer cue,129,6.4


/var/folders/xg/zxhx43fx7178bbhgcx6ksw0c0000gn/T/ipykernel_27325/2836727815.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 9. 전처리 파생본과 수동 검토 큐 저장

원본 JSON과 이미지는 수정하지 않는다. 아래 출력은 `outputs/`에 저장되며 공식 제출 파일이 아니라 모델 개발용 인덱스다.

In [10]:
quality_issue_rows = []

def add_issues(mask, issue, detail_column=None):
    columns = ["global_id", "split", "question_id"] + ([detail_column] if detail_column else [])
    for record in df.loc[mask, columns].to_dict("records"):
        quality_issue_rows.append(
            {
                "global_id": record["global_id"],
                "split": record["split"],
                "question_id": record["question_id"],
                "issue": issue,
                "detail": record.get(detail_column, "") if detail_column else "",
            }
        )


add_issues(~df["image_found"], "missing_image")
add_issues(df["image_error"].notna(), "image_open_error", "image_error")
add_issues(df["image_mb"] > 15, "large_image_over_15mib", "image_mb")
add_issues(df["image_megapixels"] > 25, "high_resolution_over_25mp", "image_megapixels")
add_issues((df["question_form"] == "LA") & (df["answer_chars"] > 250), "la_reference_over_250_chars", "answer_chars")
add_issues(df["length_constraint_match"] == False, "answer_length_constraint_mismatch")

for hash_value, group in image_hash_groups:
    ids = " | ".join(group["global_id"])
    for record in group[["global_id", "split", "question_id"]].to_dict("records"):
        quality_issue_rows.append({**record, "issue": "cross_split_identical_image", "detail": ids})

issues_df = pd.DataFrame(quality_issue_rows).sort_values(["issue", "split", "question_id"])
issues_df.to_csv(EDA_ROOT / "quality_issues.csv", index=False, encoding="utf-8-sig")

export_columns = [
    "global_id", "question_id", "split", "question_form", "image_name",
    "question", "options", "answer", "mc_choices", "requested_length", "requested_unit",
    "ocr_cue", "negation_cue", "multi_answer_cue", "image_width", "image_height",
    "image_megapixels", "image_mb", "image_format", "image_mode", "image_sha256",
]

for split in SPLITS:
    output_path = PROCESSED_ROOT / f"{split}.jsonl"
    with output_path.open("w", encoding="utf-8") as handle:
        for record in df.loc[df["split"] == split, export_columns].to_dict("records"):
            clean = {
                key: (None if not isinstance(value, (list, dict)) and pd.isna(value) else value)
                for key, value in record.items()
            }
            handle.write(json.dumps(clean, ensure_ascii=False) + "\n")

summary = {
    "records": int(len(df)),
    "split_counts": {k: int(v) for k, v in df["split"].value_counts().reindex(SPLITS).items()},
    "form_counts": {k: int(v) for k, v in df["question_form"].value_counts().reindex(FORMS).items()},
    "total_image_gib": round(float(df["image_bytes"].sum() / 1024**3), 4),
    "missing_images": int((~df["image_found"]).sum()),
    "question_chars_median": float(df["question_chars"].median()),
    "answer_chars_median_by_form": {
        form: float(labeled.loc[labeled["question_form"] == form, "answer_chars"].median())
        for form in FORMS
    },
    "ocr_cue_count": int(df["ocr_cue"].sum()),
    "negation_cue_count": int(df["negation_cue"].sum()),
    "multi_answer_cue_count": int(df["multi_answer_cue"].sum()),
    "cross_split_repeated_question_groups": int(len(question_text_groups)),
    "cross_split_identical_image_groups": int(len(image_hash_groups)),
    "large_images_over_15mib": int(len(large_images)),
    "high_resolution_over_25mp": int(len(high_resolution)),
    "la_reference_over_250_chars": int(len(la_over_250)),
    "length_constraint_mismatches": int((labeled_constraints["length_constraint_match"] == False).sum()),
    "quality_issue_rows": int(len(issues_df)),
}
(EDA_ROOT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Processed JSONL: {PROCESSED_ROOT}")
print(f"EDA summary/issues: {EDA_ROOT}")
display(pd.Series(summary, name="value").to_frame())

Processed JSONL: /Users/hyejin/AI_CV/outputs/processed
EDA summary/issues: /Users/hyejin/AI_CV/outputs/eda


,value
records,2000
split_counts,"{'train': 1000, 'validation': 200, 'test': 800}"
form_counts,"{'MC': 1036, 'SA': 512, 'LA': 452}"
total_image_gib,8.3875
missing_images,0
question_chars_median,37.0
answer_chars_median_by_form,"{'MC': 1.0, 'SA': 3.0, 'LA': 164.0}"
ocr_cue_count,331
negation_cue_count,388
multi_answer_cue_count,129


### 10. 분석 결론과 다음 실험

In [11]:
display(Markdown(f''' 
### 핵심 결론

- 데이터는 총 **{summary['records']:,}문항**, 이미지 **{summary['total_image_gib']:.2f} GiB**이며 이미지 누락은 **{summary['missing_images']}개**다.
- 문항 유형은 MC **{summary['form_counts']['MC']:,}**, SA **{summary['form_counts']['SA']:,}**, LA **{summary['form_counts']['LA']:,}**로 split 사이 비율이 거의 유지된다.
- 질문 문구상 OCR 신호가 있는 문항은 **{summary['ocr_cue_count']:,}개**지만 실제 OCR 의존도는 이보다 높을 수 있다.
- 부정형 신호가 **{summary['negation_cue_count']:,}개**, 복수 정답 신호가 **{summary['multi_answer_cue_count']:,}개**이므로 답안 라우터와 형식 검증이 중요하다.
- 교차 split 동일 이미지 SHA-256이 **{summary['cross_split_identical_image_groups']}그룹** 존재한다. 같은 이미지에 서로 다른 질문이 연결되어 있어 오류 분석과 retrieval cache 설계에서 별도로 추적해야 한다.
- 15 MiB 초과 이미지 **{summary['large_images_over_15mib']}개**, 25MP 초과 이미지 **{summary['high_resolution_over_25mp']}개**가 있어 VLM용 리사이즈와 OCR용 고해상도 파생본을 분리해야 한다.
- 공개 LA 정답 중 250자를 넘는 사례가 **{summary['la_reference_over_250_chars']}개**다. 학습 라벨은 보존하되 제출 생성 단계에서는 공식 250자 제한을 강제해야 한다.

### 다음 실험 우선순위

1. 로컬 VLM direct-answer baseline과 공식 metric scorer 구축
2. train 200개를 `visual-only / OCR / culture-knowledge / mixed`로 수작업 태깅
3. OCR 추가 전후의 validation slice 성능 비교
4. 외부 문화지식 RAG는 baseline 오류가 집중되는 문화 범주부터 구축
5. MC·SA·LA별 출력 validator를 적용해 지식 오류와 형식 오류를 분리
'''))

 
### 핵심 결론

- 데이터는 총 **2,000문항**, 이미지 **8.39 GiB**이며 이미지 누락은 **0개**다.
- 문항 유형은 MC **1,036**, SA **512**, LA **452**로 split 사이 비율이 거의 유지된다.
- 질문 문구상 OCR 신호가 있는 문항은 **331개**지만 실제 OCR 의존도는 이보다 높을 수 있다.
- 부정형 신호가 **388개**, 복수 정답 신호가 **129개**이므로 답안 라우터와 형식 검증이 중요하다.
- 교차 split 동일 이미지 SHA-256이 **6그룹** 존재한다. 같은 이미지에 서로 다른 질문이 연결되어 있어 오류 분석과 retrieval cache 설계에서 별도로 추적해야 한다.
- 15 MiB 초과 이미지 **41개**, 25MP 초과 이미지 **33개**가 있어 VLM용 리사이즈와 OCR용 고해상도 파생본을 분리해야 한다.
- 공개 LA 정답 중 250자를 넘는 사례가 **26개**다. 학습 라벨은 보존하되 제출 생성 단계에서는 공식 250자 제한을 강제해야 한다.

### 다음 실험 우선순위

1. 로컬 VLM direct-answer baseline과 공식 metric scorer 구축
2. train 200개를 `visual-only / OCR / culture-knowledge / mixed`로 수작업 태깅
3. OCR 추가 전후의 validation slice 성능 비교
4. 외부 문화지식 RAG는 baseline 오류가 집중되는 문화 범주부터 구축
5. MC·SA·LA별 출력 validator를 적용해 지식 오류와 형식 오류를 분리
